In [6]:
# =============================================================================
# comparacion_modelos_gcp.ipynb
# Comparación de modelos: Python vs GCP (AutoML + Vertex Pipelines)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# =============================================================================
# 1. MÉTRICAS DE MODELOS PYTHON (ya calculadas)
# =============================================================================

metricas_python = {
    "Modelo": [
        "Regresión Logística (Python)",
        "Random Forest (Python)",
        "XGBoost (Python)",
        "LightGBM (Python)",
        "Heurístico (Python)"
    ],
    "Recall clase 0":   [0.592, 0.072, 0.064, 0.040, 0.580],
    "Balanced Accuracy": [0.624, 0.526, 0.523, 0.512, 0.599],
    "F1 clase 0":        [0.140, 0.097, 0.090, 0.059, 0.140],
    "Plataforma":        ["Python", "Python", "Python", "Python", "Python"]
}

# =============================================================================
# 2. MÉTRICAS DE MODELOS GCP (AutoML)
# =============================================================================

metricas_gcp = {
    "Modelo": [
        "AutoML v1 (todas variables)",
        "AutoML v2 (sin variables problemáticas)",
        "AutoML v3 (dataset limpio preprocesado)"
    ],
    "Recall clase 0":    [1.00, 0.94, 1.00],
    "Balanced Accuracy": [1.00, 0.97, 1.00],
    "F1 clase 0":        [1.00, 0.97, 1.00],
    "Plataforma":        ["GCP AutoML", "GCP AutoML", "GCP AutoML"]
}

df_python = pd.DataFrame(metricas_python)
df_gcp    = pd.DataFrame(metricas_gcp)
df_total  = pd.concat([df_python, df_gcp], ignore_index=True)

print("="*60)
print("TABLA COMPARATIVA - PYTHON vs GCP")
print("="*60)
print(df_total[["Modelo", "Recall clase 0", "Balanced Accuracy", "F1 clase 0", "Plataforma"]].to_string(index=False))

TABLA COMPARATIVA - PYTHON vs GCP
                                 Modelo  Recall clase 0  Balanced Accuracy  F1 clase 0 Plataforma
           Regresión Logística (Python)           0.592              0.624       0.140     Python
                 Random Forest (Python)           0.072              0.526       0.097     Python
                       XGBoost (Python)           0.064              0.523       0.090     Python
                      LightGBM (Python)           0.040              0.512       0.059     Python
                    Heurístico (Python)           0.580              0.599       0.140     Python
            AutoML v1 (todas variables)           1.000              1.000       1.000 GCP AutoML
AutoML v2 (sin variables problemáticas)           0.940              0.970       0.970 GCP AutoML
AutoML v3 (dataset limpio preprocesado)           1.000              1.000       1.000 GCP AutoML


In [7]:
# =============================================================================
# 3. GRÁFICOS COMPARATIVOS
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Comparación de Modelos: Python vs GCP AutoML",
             fontsize=14, fontweight="bold")

metricas_grafico = ["Recall clase 0", "Balanced Accuracy", "F1 clase 0"]
colores = {"Python": "#4285F4", "GCP AutoML": "#EA4335"}

for i, metrica in enumerate(metricas_grafico):
    df_plot = df_total.dropna(subset=[metrica])
    colors  = [colores[p] for p in df_plot["Plataforma"]]

    axes[i].barh(df_plot["Modelo"], df_plot[metrica], color=colors)
    axes[i].set_title(metrica)
    axes[i].set_xlabel("Score")
    axes[i].set_xlim(0, 1.1)

    for j, val in enumerate(df_plot[metrica]):
        axes[i].text(val + 0.01, j, f"{val:.3f}", va='center', fontsize=8)

leyenda = [Patch(color="#4285F4", label="Python"),
           Patch(color="#EA4335", label="GCP AutoML")]
axes[2].legend(handles=leyenda, loc="lower right")

plt.tight_layout()
plt.savefig("comparacion_python_gcp.png", dpi=150, bbox_inches='tight')
plt.show()
print("Gráfica guardada: comparacion_python_gcp.png")

Gráfica guardada: comparacion_python_gcp.png


C:\Users\Manuela\AppData\Local\Temp\ipykernel_25376\3719722540.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# =============================================================================
# 4. ANÁLISIS DE ESTABILIDAD Y CONSISTENCIA
# =============================================================================

print("\n" + "="*60)
print("ANÁLISIS DE ESTABILIDAD Y CONSISTENCIA")
print("="*60)

print("""
1. MODELOS PYTHON:
   - Regresión Logística: recall clase 0 = 0.592 ± 0.120 (CV 10 folds)
     → Alta variabilidad (±0.120) indica inestabilidad moderada
   - Random Forest: recall clase 0 = 0.043 ± 0.037
     → Muy bajo recall pero consistente
   - XGBoost: recall clase 0 = 0.065 ± 0.053
     → Bajo recall con variabilidad moderada
   - LightGBM: recall clase 0 = 0.040 ± 0.031
     → El más consistente pero con recall más bajo

2. MODELOS GCP AutoML:
   - AutoML v1: 100% en todas las métricas → OVERFITTING
     Causa: uso de fecha_prestamo y variables correlacionadas
   - AutoML v2: 94% recall clase 0 → Más realista
     Causa: eliminación de variables problemáticas
   - AutoML v3: 100% en todas las métricas
     Nota: posible overfitting por dataset ya preprocesado y estandarizado
     El dataset df_train ya tiene variables escaladas lo que facilita
     la separación perfecta de clases para AutoML

3. CONCLUSIÓN:
   - El mejor modelo para producción es la Regresión Logística (Python)
     con recall clase 0 = 0.592 — detecta el 59.2% de morosos reales
   - AutoML v2 muestra mejor recall (94%) pero puede estar sobreajustado
   - AutoML v3 con dataset limpio logra 100% pero requiere validación
     adicional con datos completamente nuevos
   - Los modelos de Python son más interpretables y controlables
""")



ANÁLISIS DE ESTABILIDAD Y CONSISTENCIA

1. MODELOS PYTHON:
   - Regresión Logística: recall clase 0 = 0.592 ± 0.120 (CV 10 folds)
     → Alta variabilidad (±0.120) indica inestabilidad moderada
   - Random Forest: recall clase 0 = 0.043 ± 0.037
     → Muy bajo recall pero consistente
   - XGBoost: recall clase 0 = 0.065 ± 0.053
     → Bajo recall con variabilidad moderada
   - LightGBM: recall clase 0 = 0.040 ± 0.031
     → El más consistente pero con recall más bajo

2. MODELOS GCP AutoML:
   - AutoML v1: 100% en todas las métricas → OVERFITTING
     Causa: uso de fecha_prestamo y variables correlacionadas
   - AutoML v2: 94% recall clase 0 → Más realista
     Causa: eliminación de variables problemáticas
   - AutoML v3: 100% en todas las métricas
     Nota: posible overfitting por dataset ya preprocesado y estandarizado
     El dataset df_train ya tiene variables escaladas lo que facilita
     la separación perfecta de clases para AutoML

3. CONCLUSIÓN:
   - El mejor modelo para pro

In [10]:
# =============================================================================
# 5. TABLA RESUMEN FINAL
# =============================================================================

resumen = pd.DataFrame({
    "Modelo": [
        "Regresión Logística (Python)",
        "AutoML v2 (GCP)",
        "AutoML v3 (GCP - dataset limpio)"
    ],
    "Recall clase 0":    [0.592, 0.940, 1.000],
    "Balanced Accuracy": [0.624, 0.970, 1.000],
    "F1 clase 0":        [0.140, 0.970, 1.000],
    "Plataforma":        ["Python local", "GCP AutoML", "GCP AutoML"],
    "Tiempo entrenamiento": ["~5 min", "~2 horas", "~2 horas"],
    "Costo":             ["$0", "~$1-2 USD", "~$1-2 USD"],
    "Escalabilidad":     ["Limitada", "Alta", "Alta"]
})

print("="*60)
print("TABLA RESUMEN FINAL")
print("="*60)
print(resumen.to_string(index=False))

TABLA RESUMEN FINAL
                          Modelo  Recall clase 0  Balanced Accuracy  F1 clase 0   Plataforma Tiempo entrenamiento     Costo Escalabilidad
    Regresión Logística (Python)           0.592              0.624        0.14 Python local               ~5 min        $0      Limitada
                 AutoML v2 (GCP)           0.940              0.970        0.97   GCP AutoML             ~2 horas ~$1-2 USD          Alta
AutoML v3 (GCP - dataset limpio)           1.000              1.000        1.00   GCP AutoML             ~2 horas ~$1-2 USD          Alta
